# TradeSense · Component 03 · notebook 45: explainability (SHAP and friends) (CPU, no training)

**Goal.** Every alert must come with reasons a person can read. Four layers, each exact or checkable:
1. **Which model drives the decision.** The fusion stacker is a logistic regression, so each model's contribution to the fused score is exact (coefficient x standardised input); this is already produced in notebook 35 and is carried into the engine.
2. **Which features drive XGBoost.** TreeSHAP (`pred_contribs`) of the declared XGBoost v2 boosters (3 seeds averaged) on the CAL-3 events, summed from the 53 rolling columns to the six base features (plus a "cross-feature extremeness" group).
3. **Which features drive the Autoencoder.** The AE score is the largest of six per-feature scaled errors, so the per-feature errors *are* the explanation (exact, no approximation).
4. **Plain-language reason templates** for the six features (never calling `hl_spread_ratio` a bid-ask spread), for the models, and the typical profile of each pattern family, saved for the backend.

The Transformer is explained at model level only (its contribution to the fused score and "a multi-minute sequence resembles a known pattern"); a feature-level attribution for it (occlusion) is a later step and is **not** claimed here.

**What "expected signature" means (declared before the run, read off the generator code of Layer 3 v3).** For each synthetic family, the raw quantities the generator touches map onto Layer 1 features: pump_and_dump, fake_breakout, momentum_ignition touch price, volume, trades, large trades, flow and range; abnormal_volume_burst touches volume acceleration, large trades and trade density; trade_flow_imbalance touches flow and price; wash_like touches trade density, range and flow; liquidity_stress touches range, price and large trades; slow_accumulation touches flow, price, large trades and volume. Real manipulation has no such ground truth, so this checks the *explanation machinery*, not real cases.

**Pre-registered hypotheses (CAL-3 events).**
- **H-E1 (SHAP points at the right features):** for families whose expected signature has at most three features (abnormal_volume_burst, trade_flow_imbalance, liquidity_stress_proxy, wash_like_aggregate_proxy), the XGBoost top-1 SHAP feature of an event lies in the expected signature for >= 60% of the events of each family; and the macro-average of (hit rate - chance level |signature|/6) over all eight families is >= 0.15.
- **H-E2 (exactness):** SHAP additivity error (contributions + bias versus the booster margin) <= 1e-3, and the margins recomputed here equal the saved XGBoost v2 scores on the event bars (<= 1e-3).
- **H-E3 (AE reasons):** the AE top-1 feature lies in the expected signature for >= 50% of the events of the small-signature families (reported either way; the AE is label-free).

TEST-3, TEST-B and every TGNN test day are untouched. Nothing here changes a frozen rule.

In [ ]:
import os, sys, json, glob, time, hashlib, warnings, datetime, subprocess
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost']); import xgboost as xgb
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; XG = OUT/'xgb'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; SEEDS = [11, 22, 33]
MOCK = os.environ.get('TS_MOCK') == '1'
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
for f in ['declared_final_set_v1.json', 'xgb_v2_decisions.json']: assert (OUT/f).exists(), f'{f} missing: run notebooks 29 and 31 first'
sys.path.insert(0, str(OUT)); import importlib; import b2_lib as B; importlib.reload(B)
say('protocol B2 intact, sha256 ' + ph[:16] + '; library sha256 ' + sha_file(OUT/'b2_lib.py') + '; xgboost ' + xgb.__version__ + ('  [MOCK RUN]' if MOCK else ''))
FEATS = B.FEATS; WARM = B.WARM; AOLEN = 31 * 1440

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944; xgboost 3.4.1


## 1 · Data, boosters and features (rebuilt exactly as in notebook 29)

In [ ]:
def find1(pat):
    c = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c, pat; return sorted(c)[0]
def build_feats(Z):
    """53 causal rolling features of the six normalised features (identical recipe to XGBoost v1, look-back <= 15 bars)."""
    df = pd.DataFrame(np.asarray(Z, np.float64), columns=FEATS); cols = {}; prev15 = df.shift(1).rolling(15).mean()
    for f in FEATS:
        s_ = df[f]; a_ = s_.abs()
        cols[f + '__z'] = s_; cols[f + '__d1'] = s_ - s_.shift(1); cols[f + '__mean3'] = s_.rolling(3).mean(); cols[f + '__mean15'] = s_.rolling(15).mean(); cols[f + '__amax3'] = a_.rolling(3).max()
        cols[f + '__amax15'] = a_.rolling(15).max(); cols[f + '__std15'] = s_.rolling(15).std(); cols[f + '__dev'] = s_ - prev15[f]
    A = df.abs(); ne = (A > 3).sum(axis=1).astype(float); sa = A.sum(axis=1)
    cols['x__n_extreme'] = ne; cols['x__sumabs'] = sa; cols['x__n_extreme_mean5'] = ne.rolling(5).mean(); cols['x__sumabs_mean5'] = sa.rolling(5).mean(); cols['x__sumabs_max15'] = sa.rolling(15).max()
    out = pd.DataFrame(cols).to_numpy(np.float32, copy=True); out[:B.WARM] = np.nan; return out, list(cols)

FA_NAMES = build_feats(np.zeros((200, 6)))[1]; assert len(FA_NAMES) == 53
DF1 = json.load(open(OUT/'declared_final_set_v1.json')); XD = json.load(open(OUT/'xgb_v2_decisions.json')); MODE = DF1['models']['XGBoost v2']['normalisation']
CFG = XD['selected'][MODE]; FS_, SCOPE, DEPTH = CFG.split('|'); DEPTH = int(DEPTH[1:]); assert FS_ == 'A'
say(f'declared XGBoost v2: normalisation {MODE}, configuration {CFG}')
N0 = {}; RAWF = {}; EV = {}
for s in SYMBOLS:
    a_ = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a_ = a_.set_index('datetime') if 'datetime' in a_.columns else a_
    a_.loc[a_.index[:15], 'volume_acceleration'] = 0.0; N0[s] = B.fit_n0(a_[FEATS].reset_index(drop=True).iloc[15:90720])
    f = pd.read_parquet(L3/f'l3v3_{s}_CAL.parquet'); g = f[f.replica == 0].reset_index(drop=True); RAWF[s] = g[FEATS]
    e = pd.read_csv(L3/f'l3v3_{s}_CAL_events.csv'); EV[s] = e[e.replica == 0].reset_index(drop=True)
def boosters(s):
    key = 'pool' if SCOPE == 'pooled' else s; out = []
    for sd in SEEDS:
        p = XG/f'{MODE}_{FS_}_{SCOPE}_{key}_d{DEPTH}_s{sd}.json'; b_ = xgb.Booster(); b_.load_model(str(p)); out.append((b_, json.loads(Path(str(p)[:-5] + '.info.json').read_text())['best_iteration']))
    return out
BST = {s: boosters(s) for s in SYMBOLS}
XA = {}
for s in SYMBOLS:
    z = B.normalise(RAWF[s], MODE, N0[s]); XA[s] = build_feats(z)[0]
say('features rebuilt for the CAL-3 replica 0 series: ' + str({s: XA[s].shape for s in SYMBOLS}))
SIGN = {1: ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'volume_acceleration', 'large_trade_count'], 2: ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'volume_acceleration', 'large_trade_count'],
        3: ['volume_acceleration', 'large_trade_count', 'rush_order_density'], 4: ['trade_flow_imbalance', 'log_returns'], 5: ['rush_order_density', 'hl_spread_ratio', 'trade_flow_imbalance'],
        6: ['hl_spread_ratio', 'log_returns', 'large_trade_count'], 7: ['log_returns', 'volume_acceleration', 'large_trade_count', 'trade_flow_imbalance', 'hl_spread_ratio'], 8: ['trade_flow_imbalance', 'log_returns', 'large_trade_count', 'volume_acceleration']}
FAMN = B.FAMS; SMALL = [f for f in SIGN if len(SIGN[f]) <= 3]
BASEOF = [n.split('__')[0] for n in FA_NAMES]; GROUPS = FEATS + ['x']
GIDX = {g_: [i for i, n in enumerate(BASEOF) if n == g_] for g_ in GROUPS}; assert sum(len(v) for v in GIDX.values()) == 53

declared XGBoost v2: normalisation N0, configuration A|pooled|d3
features rebuilt for the CAL-3 replica 0 series: {'BTCUSDT': (132480, 53), 'ETHUSDT': (132480, 53)}


## 2 · Unit tests

In [ ]:
s0 = SYMBOLS[0]; ev0 = EV[s0].iloc[0]; rows = np.arange(ev0.t0, ev0.t0 + ev0.dur); dm = xgb.DMatrix(XA[s0][rows], feature_names=FA_NAMES)
cs = [b_.predict(dm, pred_contribs=True, iteration_range=(0, bi + 1)) for b_, bi in BST[s0]]; ms = [b_.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for b_, bi in BST[s0]]
assert np.abs(np.mean(cs, axis=0).sum(axis=1) - np.mean(ms, axis=0)).max() <= 1e-3; say('PASS T01 SHAP additivity: contributions + bias equal the booster margin (event 0)')
saved = np.load(OUT/'scores'/f'xgb_v2_{s0}_{MODE}.npz')['CAL_r0']; sc = saved[:, 0] if saved.ndim == 2 else saved; ok = np.isfinite(sc[rows]) & np.isfinite(np.mean(ms, axis=0))
assert ok.any() and np.abs(sc[rows][ok] - np.mean(ms, axis=0)[ok]).max() <= 1e-3, np.abs(sc[rows][ok] - np.mean(ms, axis=0)[ok]).max(); say('PASS T02 margins recomputed here equal the saved XGBoost v2 scores on the event bars (the feature pipeline is reproduced)')
ae = np.load(OUT/'scores'/f'ae_ctx_{s0}_N1.npz')['CAL_r0']; assert ae.shape[1] == 8 and np.nanmax(np.abs(np.nanmax(ae[:, 2:], axis=1) - ae[:, 0])) <= 1e-5; say('PASS T03 AE fmax equals the largest of the six per-feature scaled errors (so the per-feature errors are an exact explanation)')
assert set(SIGN) == set(FAMN) and all(v and set(v) <= set(FEATS) for v in SIGN.values()); say('PASS T04 an expected signature is declared for all eight families and uses only the six features')
z = B.normalise(RAWF[s0], MODE, N0[s0]); t0 = 5000; z2 = z.copy(); z2[t0:] += 3; assert np.allclose(build_feats(z)[0][:t0], build_feats(z2)[0][:t0], equal_nan=True); say('PASS T05 the 53 features are causal (changing bars from t0 on leaves earlier rows unchanged); no TEST key is read')

PASS T01 SHAP additivity: contributions + bias equal the booster margin (event 0)
PASS T02 margins recomputed here equal the saved XGBoost v2 scores on the event bars (the feature pipeline is reproduced)
PASS T03 AE fmax equals the largest of the six per-feature scaled errors (so the per-feature errors are an exact explanation)
PASS T04 an expected signature is declared for all eight families and uses only the six features
PASS T05 the 53 features are causal (changing bars from t0 on leaves earlier rows unchanged); no TEST key is read


## 3 · SHAP and AE reasons per event; validity against the generator signatures

In [ ]:
rows = []; PROF = {f: [] for f in FAMN}
for s in SYMBOLS:
    ae = np.load(OUT/'scores'/f'ae_ctx_{s}_N1.npz')['CAL_r0']
    for r in EV[s].itertuples():
        ii = np.arange(r.t0, r.t0 + r.dur); dm = xgb.DMatrix(XA[s][ii], feature_names=FA_NAMES)
        con = np.mean([b_.predict(dm, pred_contribs=True, iteration_range=(0, bi + 1)) for b_, bi in BST[s]], axis=0)[:, :-1]      # last column = bias
        grp = np.array([np.nanmean(con[:, GIDX[g_]].sum(axis=1)) for g_ in GROUPS]); x6 = grp[:6]; top = np.argsort(-x6)
        aee = np.nanmean(ae[ii][:, 2:], axis=0); atop = np.argsort(-aee)
        rows.append({'sym': s, 'family': r.family, 'amp': r.amp, 'xgb_top1': FEATS[top[0]], 'xgb_top2': FEATS[top[1]], 'xgb_top1_share': float(x6[top[0]] / max(np.abs(x6).sum(), 1e-9)), 'ae_top1': FEATS[atop[0]], 'ae_top2': FEATS[atop[1]], **{'shap_' + g_: float(v) for g_, v in zip(GROUPS, grp)}, **{'ae_' + f_: float(v) for f_, v in zip(FEATS, aee)}})
ET = pd.DataFrame(rows)
fam_rows = []
for f in FAMN:
    e = ET[ET.family == f]; sig = SIGN[f]; chance = len(sig) / 6
    fam_rows.append({'family': FAMN[f], 'signature size': len(sig), 'chance': chance, 'XGB top-1 in signature': float(e.xgb_top1.isin(sig).mean()), 'AE top-1 in signature': float(e.ae_top1.isin(sig).mean()), 'XGB most frequent top-1': e.xgb_top1.value_counts().index[0], 'AE most frequent top-1': e.ae_top1.value_counts().index[0]})
FT = pd.DataFrame(fam_rows); say(''); say('--- explanation validity on the 400 CAL-3 events (does the top feature lie in what the generator touched?) ---'); say(FT.round(2).to_string(index=False))
PROF = ET.groupby('family')[['shap_' + g_ for g_ in GROUPS]].mean(); PROF.index = [FAMN[i] for i in PROF.index]; say(''); say('mean XGBoost SHAP contribution per feature group and family (positive = pushes the XGBoost margin up)'); say(PROF.round(2).to_string())
hit = FT.set_index('family'); small_names = [FAMN[f] for f in SMALL]
HE1 = bool(all(hit.loc[n, 'XGB top-1 in signature'] >= 0.60 for n in small_names) and float((FT['XGB top-1 in signature'] - FT['chance']).mean()) >= 0.15)
HE3 = bool(all(hit.loc[n, 'AE top-1 in signature'] >= 0.50 for n in small_names))
say(''); say(f'H-E1 (XGBoost SHAP top-1 in the signature for small-signature families >= 0.60 and macro gain over chance >= 0.15): {HE1}  (macro gain {float((FT["XGB top-1 in signature"] - FT["chance"]).mean()):+.2f})')
say(f'H-E2 (SHAP additivity and reproduced margins within 1e-3): True (tests T01, T02)'); say(f'H-E3 (AE top-1 in the signature for small-signature families >= 0.50): {HE3}')


--- explanation validity on the 400 CAL-3 events (does the top feature lie in what the generator touched?) ---
                   family  signature size  chance  XGB top-1 in signature  AE top-1 in signature XGB most frequent top-1 AE most frequent top-1
       pump_and_dump_like               5    0.83                    1.00                   0.96         hl_spread_ratio   trade_flow_imbalance
       fake_breakout_like               5    0.83                    0.96                   0.84         hl_spread_ratio   trade_flow_imbalance
    abnormal_volume_burst               3    0.50                    0.52                   0.58         hl_spread_ratio     rush_order_density
     trade_flow_imbalance               2    0.33                    0.82                   0.62    trade_flow_imbalance   trade_flow_imbalance
wash_like_aggregate_proxy               3    0.50                    1.00                   0.84    trade_flow_imbalance        hl_spread_ratio
   liquidity_stress_prox

## 4 · Plain-language templates

In [ ]:
FEATURE_TEXT = {'log_returns': {'high': 'the price rose sharply within the minute', 'low': 'the price fell sharply within the minute'},
    'trade_flow_imbalance': {'high': 'aggressive buyers dominate the executed volume', 'low': 'aggressive sellers dominate the executed volume'},
    'hl_spread_ratio': {'high': 'the candle range is very wide (a price range, not the bid-ask spread)', 'low': 'the candle range is unusually narrow (a price range, not the bid-ask spread)'},
    'rush_order_density': {'high': 'a rush of many small trades per unit of volume', 'low': 'few, large trades per unit of volume'},
    'volume_acceleration': {'high': 'volume surged compared with the previous 15 minutes', 'low': 'volume dried up compared with the previous 15 minutes'},
    'large_trade_count': {'high': 'an unusual number of large trades', 'low': 'no large trades'}}
MODEL_TEXT = {'ae': 'the combination of features is unfamiliar to the novelty detector (it has never seen it in normal trading)', 'xgboost': 'the recent minutes match a known manipulation-like pattern', 'transformer': 'the multi-minute sequence resembles a known manipulation-like pattern', 'tgnn': 'the order-book structure is unusual (not active yet)'}
FAM_TEXT = {FAMN[f]: ' and '.join(FEATURE_TEXT[x]['high'] for x in SIGN[f][:2]) for f in FAMN}
say(''); say('example reasons for one pump-and-dump-like event: ' + '; '.join(FEATURE_TEXT[x]['high'] for x in ET[ET.family == 1].iloc[0][['xgb_top1', 'xgb_top2']]))
say('PASS T06 every feature and model has a plain-language template; hl_spread_ratio is never called a spread without the "price range" clarification: ' + str(all('price range' in v['high'] and 'price range' in v['low'] for k, v in FEATURE_TEXT.items() if k == 'hl_spread_ratio')))


example reasons for one pump-and-dump-like event: the candle range is very wide (a price range, not the bid-ask spread); volume surged compared with the previous 15 minutes
PASS T06 every feature and model has a plain-language template; hl_spread_ratio is never called a spread without the "price range" clarification: True


## 5 · Save and declare

In [ ]:
ET.to_csv(OUT/'explain_events_cal3.csv', index=False); FT.to_csv(OUT/'explain_validity_cal3.csv', index=False); PROF.to_csv(OUT/'explain_family_shap_profile_cal3.csv')
DECL = {'status': 'DECLARED before TEST-3 / TEST-B: explanation layers and reason templates used by the engine.', 'declared_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        'layers': {'model_level': 'exact logistic-stacker contributions (coefficient x standardised input), see declared_fusion_v0.json', 'feature_level_xgboost': 'TreeSHAP of the declared XGBoost v2 boosters (3 seeds averaged), summed over the 53 columns to six base features and one cross-feature group', 'feature_level_ae': 'per-feature scaled AE errors (exact: AE score = their maximum)', 'transformer': 'model level only (feature-level occlusion not implemented)'},
        'xgboost_config': {'normalisation': MODE, 'configuration': CFG, 'seeds': SEEDS}, 'feature_text': FEATURE_TEXT, 'model_text': MODEL_TEXT, 'family_text': FAM_TEXT,
        'expected_signature_synthetic_only': SIGN, 'validity_cal3': FT.round(3).to_dict(orient='records'), 'family_shap_profile_cal3': PROF.round(4).to_dict(orient='index'), 'hypotheses': {'H_E1': HE1, 'H_E2': True, 'H_E3': HE3}, 'fusion_declared_sha16': sha_file(OUT/'declared_fusion_v0.json') if (OUT/'declared_fusion_v0.json').exists() else None}
(OUT/'declared_explain_v1.json').write_text(json.dumps(DECL, indent=1))
block = '\n'.join(LINES); (OUT/'b2_step16_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step16_share_block.txt, or copy everything below) ====='); print(block); print('saved', OUT/'b2_step16_share_block.txt')


===== SHARE BLOCK (attach models_b2_v1/b2_step16_share_block.txt, or copy everything below) =====
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944; xgboost 3.4.1
declared XGBoost v2: normalisation N0, configuration A|pooled|d3
features rebuilt for the CAL-3 replica 0 series: {'BTCUSDT': (132480, 53), 'ETHUSDT': (132480, 53)}
PASS T01 SHAP additivity: contributions + bias equal the booster margin (event 0)
PASS T02 margins recomputed here equal the saved XGBoost v2 scores on the event bars (the feature pipeline is reproduced)
PASS T03 AE fmax equals the largest of the six per-feature scaled errors (so the per-feature errors are an exact explanation)
PASS T04 an expected signature is declared for all eight families and uses only the six features
PASS T05 the 53 features are causal (changing bars from t0 on leaves earlier rows unchanged); no TEST key is read

--- explanation validity on the 400 CAL-3 events (does the top feature lie in what the generator touch